In [1]:
# Cell 1: Install dependencies
!pip install -q scipy numpy

In [2]:
# Cell 2: Load GitHub token
import os
try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    print("GitHub token loaded from Colab Secrets.")
except Exception:
    token = input("Enter your GitHub token: ")
    os.environ["GITHUB_TOKEN"] = token

GitHub token loaded from Colab Secrets.


In [3]:
# Cell 3: Imports
import json
import time
import datetime
import requests
import numpy as np
from scipy.stats import spearmanr

HEADERS = {
    "Accept": "application/vnd.github+json",
    "Authorization": f"Bearer {os.environ.get('GITHUB_TOKEN', '')}",
    "X-GitHub-Api-Version": "2022-11-28",
}

REVERT_KEYWORDS     = ["revert", "revert pr", "reverts"]
BUGFIX_KEYWORDS   = ["fix", "bugfix", "bug fix", "hotfix", "patch"]
SECURITY_KEYWORDS = ["security", "cve", "vuln", "vulnerability"]
OUTCOME_WINDOW_DAYS = 60

MODELS = [
    "qwen2.5:7b-instruct",
    "llama3.1:8b",
    "gemma3:4b",
]

DISPLAY_NAMES = {
    "qwen2.5:7b-instruct": "Qwen2.5-7B",
    "llama3.1:8b":         "Llama3.1-8B",
    "gemma3:4b":           "Gemma3-4B",
}

print("Imports done.")

Imports done.


In [4]:
# Cell 4: Upload files
from google.colab import files

print("Upload pr_dataset.json:")
up1 = files.upload()
PR_DATASET_PATH = list(up1.keys())[0]
print(f"  Using: {PR_DATASET_PATH}")

print("\nUpload llm_judge_results.json (from RQ1):")
up2 = files.upload()
RQ1_RESULTS_PATH = list(up2.keys())[0]
print(f"  Using: {RQ1_RESULTS_PATH}")

Upload pr_dataset.json:


Saving pr_dataset_django.json to pr_dataset_django.json
  Using: pr_dataset_django.json

Upload llm_judge_results.json (from RQ1):


Saving llm_judge_results_django.json to llm_judge_results_django.json
  Using: llm_judge_results_django.json


In [6]:
# Cell 5: GitHub API helpers

def parse_github_dt(dt_str: str) -> datetime.datetime:
    return datetime.datetime.strptime(dt_str, "%Y-%m-%dT%H:%M:%SZ")


def get_commits_after_merge(merged_at: str, repo: str, retries: int = 3) -> list:
    merge_dt  = parse_github_dt(merged_at)
    cutoff_dt = merge_dt + datetime.timedelta(days=OUTCOME_WINDOW_DAYS)
    since_str = merge_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    until_str = cutoff_dt.strftime("%Y-%m-%dT%H:%M:%SZ")

    url     = f"https://api.github.com/repos/{repo}/commits"
    commits = []
    page    = 1

    while True:
        params = {
            "since": since_str, "until": until_str,
            "per_page": 100,    "page": page,
        }
        for attempt in range(retries):
            try:
                resp = requests.get(url, headers=HEADERS, params=params, timeout=30)

                # handle secondary rate limit — read Retry-After header
                if resp.status_code == 403:
                    retry_after = int(resp.headers.get("Retry-After", 60))
                    print(f"    Secondary rate limit hit. Waiting {retry_after}s...")
                    time.sleep(retry_after + 5)
                    continue  # retry after waiting

                resp.raise_for_status()
                batch = resp.json()
                break
            except Exception as e:
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)
                else:
                    batch = []
                    break

        if not batch:
            break

        for c in batch:
            commits.append({
                "sha":     c["sha"],
                "message": (c.get("commit") or {}).get("message", "").lower(),
            })

        if len(batch) < 100:
            break
        page += 1
        time.sleep(0.5)

    return commits

def check_and_wait_rate_limit():
    """Check remaining rate limit and wait if running low."""
    try:
        resp = requests.get(
            "https://api.github.com/rate_limit",
            headers=HEADERS, timeout=10
        )
        data = resp.json()
        remaining = data["rate"]["remaining"]
        reset_time = data["rate"]["reset"]
        print(f"    Rate limit: {remaining} remaining")
        if remaining < 100:
            wait = reset_time - time.time() + 10
            print(f"    Rate limit low — waiting {wait:.0f}s...")
            time.sleep(max(wait, 0))
    except Exception:
        time.sleep(60)


def message_matches(message: str, keywords: list) -> bool:
    return any(kw in message for kw in keywords)


def pr_number_in_message(message: str, pr_number: int) -> bool:
    return f"#{pr_number}" in message or f"gh-{pr_number}" in message.lower()


def get_commit_files(sha: str, repo: str, retries: int = 3) -> list:
    url = f"https://api.github.com/repos/{repo}/commits/{sha}"
    for attempt in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            return [f["filename"] for f in resp.json().get("files", [])]
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    return []


def extract_pr_files_from_diff(diff: str) -> list:
    files = []
    for line in diff.splitlines():
        if line.startswith("+++ b/"):
            fname = line[6:].strip()
            if fname and fname not in files:
                files.append(fname)
    return files


def files_overlap(commit_files: list, pr_files: list) -> bool:
    return any(f in set(pr_files) for f in commit_files)


def compute_outcome_label(pr: dict, commits: list, repo: str) -> dict:
    pr_number = pr.get("pr_number")
    pr_files  = extract_pr_files_from_diff(pr.get("diff", ""))

    for commit in commits:
        msg = commit["message"]
        sha = commit["sha"]

        if message_matches(msg, REVERT_KEYWORDS):
            if pr_number and pr_number_in_message(msg, pr_number):
                return {"outcome": 1, "trigger": "revert",
                        "commit_sha": sha, "commit_message": msg[:120]}

        if message_matches(msg, BUGFIX_KEYWORDS) and pr_files:
            commit_files = get_commit_files(sha, repo)
            if files_overlap(commit_files, pr_files):
                return {"outcome": 1, "trigger": "bugfix",
                        "commit_sha": sha, "commit_message": msg[:120]}
            time.sleep(0.2)

        if message_matches(msg, SECURITY_KEYWORDS) and pr_files:
            commit_files = get_commit_files(sha, repo)
            if files_overlap(commit_files, pr_files):
                return {"outcome": 1, "trigger": "security",
                        "commit_sha": sha, "commit_message": msg[:120]}
            time.sleep(0.2)

    return {"outcome": 0, "trigger": None,
            "commit_sha": None, "commit_message": None}


print("API helpers defined.")

API helpers defined.


In [7]:
# Cell 6: Collect outcome labels
#
# This makes one API call per PR to fetch commit history.
# For 500 PRs expect ~10-15 minutes depending on rate limit.
# Checkpoints every 50 PRs so you can resume if interrupted.
#
# To re-run collection even if rq3_outcome_labels.json exists,
# delete the file or set FORCE_RECOLLECT = True below.

FORCE_RECOLLECT    = True
OUTCOME_LABELS_PATH = "rq3_outcome_labels.json"

import os as _os
if not _os.path.exists(OUTCOME_LABELS_PATH) or FORCE_RECOLLECT:

    with open(PR_DATASET_PATH, encoding="utf-8") as f:
        pr_dataset = json.load(f)

    # Auto-detect repo from dataset
    repo = pr_dataset[0].get("repo", "pandas-dev/pandas")
    print(f"Repo: {repo}")
    print(f"PRs to label: {len(pr_dataset)}")

    # Resume from checkpoint if available
    try:
        with open("rq3_checkpoint.json", encoding="utf-8") as f:
            labeled = json.load(f)
        done_ids = {r["pr_id"] for r in labeled}
        print(f"Resuming from checkpoint: {len(labeled)} already done")
    except FileNotFoundError:
        labeled, done_ids = [], set()

    for i, pr in enumerate(pr_dataset):
        pr_id     = pr.get("pr_id")
        pr_number = pr.get("pr_number")
        merged_at = pr.get("merged_at")

        if pr_id in done_ids:
            continue

        if not merged_at:
            print(f"  PR {pr_id}: missing merged_at, skipping")
            labeled.append({"pr_id": pr_id, "pr_number": pr_number,
                             "outcome": None, "trigger": None})
            continue

        if not pr_number:
            print(f"  PR {pr_id}: missing pr_number, skipping")
            labeled.append({"pr_id": pr_id, "pr_number": pr_number,
                             "outcome": None, "trigger": None})
            continue

        print(f"  Labeling PR {i+1}/{len(pr_dataset)}: #{pr_number}")
        commits = get_commits_after_merge(merged_at, repo)
        result  = compute_outcome_label(pr, commits, repo)

        labeled.append({
            "pr_id":          pr_id,
            "pr_number":      pr_number,
            "merged_at":      merged_at,
            "outcome":        result["outcome"],
            "trigger":        result["trigger"],
            "commit_sha":     result["commit_sha"],
            "commit_message": result["commit_message"],
        })
        done_ids.add(pr_id)
        time.sleep(3)

        if len(labeled) % 50 == 0:
            with open("rq3_checkpoint.json", "w", encoding="utf-8") as f:
                json.dump(labeled, f, indent=2)
            print(f"  Checkpoint: {len(labeled)} PRs labeled")

    with open(OUTCOME_LABELS_PATH, "w", encoding="utf-8") as f:
        json.dump(labeled, f, indent=2, ensure_ascii=False)

    n_pos = sum(1 for r in labeled if r["outcome"] == 1)
    print(f"\nLabeled  : {len(labeled)} PRs")
    print(f"Reverted : {n_pos} ({n_pos/max(len(labeled),1)*100:.1f}%)")

else:
    print(f"Loading existing labels from {OUTCOME_LABELS_PATH}")
    with open(OUTCOME_LABELS_PATH, encoding="utf-8") as f:
        labeled = json.load(f)
    n_pos = sum(1 for r in labeled if r["outcome"] == 1)
    print(f"Loaded {len(labeled)} labels — {n_pos} positive")

Repo: django/django
PRs to label: 500
  Labeling PR 1/500: #21075
  Labeling PR 2/500: #21070
  Labeling PR 3/500: #21056
  Labeling PR 4/500: #21051
  Labeling PR 5/500: #21050
  Labeling PR 6/500: #21046
  Labeling PR 7/500: #21045
  Labeling PR 8/500: #21036
  Labeling PR 9/500: #21035
  Labeling PR 10/500: #21029
  Labeling PR 11/500: #20998
  Labeling PR 12/500: #20964
  Labeling PR 13/500: #20960
  Labeling PR 14/500: #20948
  Labeling PR 15/500: #20911
  Labeling PR 16/500: #20901
  Labeling PR 17/500: #20900
  Labeling PR 18/500: #20890
  Labeling PR 19/500: #20889
  Labeling PR 20/500: #20882
  Labeling PR 21/500: #20881
  Labeling PR 22/500: #20877
  Labeling PR 23/500: #20864
  Labeling PR 24/500: #20856
  Labeling PR 25/500: #20854
  Labeling PR 26/500: #20852
  Labeling PR 27/500: #20837
  Labeling PR 28/500: #20831
  Labeling PR 29/500: #20828
  Labeling PR 30/500: #20823
  Labeling PR 31/500: #20807
  Labeling PR 32/500: #20792
  Labeling PR 33/500: #20789
  Labeling PR 

In [8]:
# Cell 7: Feature-weighted baseline helper
#
# Weights from RQ1 circularity check (comment-only GT):
#   additions   rho=0.216 -> 0.40
#   max_nesting rho=0.119 -> 0.25
#   deletions   rho=0.100 -> 0.20
#   cyclomatic  rho=0.095 -> 0.15

def min_max_normalize(values):
    mn, mx = min(values), max(values)
    if mx == mn:
        return [0.0] * len(values)
    return [(v - mn) / (mx - mn) for v in values]


def compute_composite_baseline(records):
    n_add = min_max_normalize([r.get("additions", 0)        for r in records])
    n_nd  = min_max_normalize([r.get("max_nesting", 0)       for r in records])
    n_del = min_max_normalize([r.get("deletions", 0)         for r in records])
    n_cc  = min_max_normalize([r.get("cyclomatic_delta", 0)  for r in records])
    return [0.40*a + 0.25*n + 0.20*d + 0.15*c
            for a, n, d, c in zip(n_add, n_nd, n_del, n_cc)]


def spearman(x, y):
    rho, p = spearmanr(x, y)
    return {
        "spearman_rho": round(float(rho), 4),
        "p_value":      round(float(p), 4),
        "significant":  bool(p < 0.05),
    }


def compute_per_feature_correlations(records, outcomes):
    features = {
        "additions":       [r.get("additions", 0)        for r in records],
        "deletions":       [r.get("deletions", 0)        for r in records],
        "cyclomatic_delta":[r.get("cyclomatic_delta", 0) for r in records],
        "max_nesting":     [r.get("max_nesting", 0)      for r in records],
        "logic_density":   [r.get("logic_density", 0.0)  for r in records],
    }
    return {name: spearman(vals, outcomes) for name, vals in features.items()}


print("Baseline helpers defined.")

Baseline helpers defined.


In [9]:
# Cell 8: Evaluate — Spearman correlation between LLM scores and outcomes

with open(PR_DATASET_PATH, encoding="utf-8") as f:
    pr_dataset = json.load(f)
pr_lookup = {pr["pr_id"]: pr for pr in pr_dataset}

with open(RQ1_RESULTS_PATH, encoding="utf-8") as f:
    rq1_data = json.load(f)
rq1_results = rq1_data.get("results", [])

outcome_lookup = {
    r["pr_id"]: r["outcome"]
    for r in labeled
    if r.get("outcome") is not None
}
n_pos = sum(1 for v in outcome_lookup.values() if v == 1)
n_neg = sum(1 for v in outcome_lookup.values() if v == 0)
print(f"Outcome labels: {len(outcome_lookup)} PRs ({n_pos} reverted, {n_neg} not)")

all_metrics = {}

for model_name in MODELS:
    print(f"\n{'='*55}")
    print(f"RQ3: {DISPLAY_NAMES.get(model_name, model_name)}")
    print(f"{'='*55}")

    model_results = [
        r for r in rq1_results
        if r["model_name"] == model_name
        and isinstance(r.get("risk_score"), int)
        and r.get("pr_id") in outcome_lookup
    ]

    if len(model_results) < 10:
        print(f"  Not enough PRs ({len(model_results)}). Skipping.")
        all_metrics[model_name] = {"note": "Insufficient data", "n": len(model_results)}
        continue

    print(f"  PRs with LLM score + outcome: {len(model_results)}")

    llm_scores     = [r["risk_score"]            for r in model_results]
    outcome_labels = [outcome_lookup[r["pr_id"]] for r in model_results]

    # Build feature records — try rq1_results first, fallback to pr_dataset
    feature_records = []
    for r in model_results:
        pr = pr_lookup.get(r["pr_id"], {})
        feature_records.append({
            "additions":        r.get("additions",        pr.get("additions", 0)),
            "deletions":        r.get("deletions",        pr.get("deletions", 0)),
            "cyclomatic_delta": r.get("cyclomatic_delta", pr.get("cyclomatic_delta_total", 0)),
            "max_nesting":      r.get("max_nesting",      pr.get("max_nesting_depth", 0)),
            "logic_density":    r.get("logic_density",    pr.get("logic_density_total", 0.0)),
        })

    composite_scores = compute_composite_baseline(feature_records)
    llm_corr         = spearman(llm_scores, outcome_labels)
    composite_corr   = spearman(composite_scores, outcome_labels)
    feature_corrs    = compute_per_feature_correlations(feature_records, outcome_labels)

    print(f"\n  {'Signal':<28} {'Spearman rho':>12} {'p-value':>10} {'Sig':>6}")
    print(f"  {'-'*60}")
    print(f"  {'LLM Risk Score':<28} "
          f"{llm_corr['spearman_rho']:>12.4f} "
          f"{llm_corr['p_value']:>10.4f} "
          f"{'Yes' if llm_corr['significant'] else 'No':>6}")
    print(f"  {'Feature-Weighted Baseline':<28} "
          f"{composite_corr['spearman_rho']:>12.4f} "
          f"{composite_corr['p_value']:>10.4f} "
          f"{'Yes' if composite_corr['significant'] else 'No':>6}")
    print(f"\n  Per-feature:")
    for fname, fc in feature_corrs.items():
        print(f"    {fname:<26} rho={fc['spearman_rho']:>7.4f} "
              f"p={fc['p_value']:.4f} {'*' if fc['significant'] else ''}")

    llm_rho  = llm_corr["spearman_rho"]
    comp_rho = composite_corr["spearman_rho"]
    delta    = llm_rho - comp_rho
    if delta > 0:
        print(f"\n  LLM outperforms baseline by delta_rho={delta:.4f}")
    elif delta == 0:
        print(f"\n  LLM matches baseline")
    else:
        print(f"\n  Baseline outperforms LLM by delta_rho={abs(delta):.4f}")

    all_metrics[model_name] = {
        "rq3":                              "Does AI-generated risk ranking align with actual review outcomes?",
        "gt_method":                        "revert commit referencing PR number within 60 days",
        "n_prs":                            len(model_results),
        "n_positive_outcomes":              sum(outcome_labels),
        "n_negative_outcomes":              len(outcome_labels) - sum(outcome_labels),
        "llm_risk_score_correlation":       llm_corr,
        "feature_weighted_baseline_correlation": composite_corr,
        "baseline_weights":                 {"additions": 0.40, "max_nesting": 0.25,
                                             "deletions": 0.20, "cyclomatic_delta": 0.15},
        "per_feature_correlations":         feature_corrs,
        "llm_vs_baseline_delta_rho":        round(delta, 4),
    }

print("\nEvaluation complete.")

Outcome labels: 500 PRs (140 reverted, 360 not)

RQ3: Qwen2.5-7B
  PRs with LLM score + outcome: 500

  Signal                       Spearman rho    p-value    Sig
  ------------------------------------------------------------
  LLM Risk Score                    -0.0835     0.0622     No
  Feature-Weighted Baseline         -0.0140     0.7546     No

  Per-feature:
    additions                  rho=-0.0347 p=0.4384 
    deletions                  rho=-0.0061 p=0.8923 
    cyclomatic_delta           rho=-0.0570 p=0.2034 
    max_nesting                rho= 0.0257 p=0.5659 
    logic_density              rho=-0.0204 p=0.6483 

  Baseline outperforms LLM by delta_rho=0.0695

RQ3: Llama3.1-8B
  PRs with LLM score + outcome: 500

  Signal                       Spearman rho    p-value    Sig
  ------------------------------------------------------------
  LLM Risk Score                    -0.0113     0.8012     No
  Feature-Weighted Baseline         -0.0140     0.7546     No

  Per-feature:


In [10]:
# Cell 9: Save and download results

output = {
    "metrics": all_metrics,
    "outcome_summary": {
        "total_prs_labeled": len(outcome_lookup),
        "positive":          n_pos,
        "negative":          n_neg,
        "window_days":       OUTCOME_WINDOW_DAYS,
        "gt_method":         "revert-only",
    },
}

out_filename = "llm_judge_rq3_results.json"
with open(out_filename, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"Results saved to {out_filename}")

# Also download outcome labels for reuse
from google.colab import files
files.download(out_filename)
files.download(OUTCOME_LABELS_PATH)

Results saved to llm_judge_rq3_results.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>